# Strategy Explorer — Volume Profile Bot

This notebook lets you **see** the strategy in action, understand each filter, and tune parameters interactively.

## What you can do here
1. Load and explore 23 years of EURUSD data
2. Visualize the volume profile on any date/window
3. Walk through a specific trade signal step-by-step
4. Analyse the trade journal from the last backtest
5. Compare performance across years

---
Run cells from top to bottom. Each section is independent after setup.

## 0 — Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # add project root to path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Dark theme so charts look like a real trading terminal
plt.style.use('dark_background')
BG     = '#0f0f1a'
PANEL  = '#16162a'
GREEN  = '#1D9E75'
RED    = '#E24B4A'
BLUE   = '#378ADD'
ORANGE = '#D85A30'
TEXT   = '#c8c8c4'
GRAY   = '#888780'

from src.data.csv_provider import CSVProvider
from src.config import Config

provider = CSVProvider()
print('Setup complete.')

## 1 — Load Data

In [ ]:
# Load both timeframes
df_h1  = provider.get_ohlcv(symbol='EURUSD', timeframe='H1')
df_m15 = provider.get_ohlcv(symbol='EURUSD', timeframe='M15')

print(f'H1  bars: {len(df_h1):,}  |  {df_h1.index[0].date()} → {df_h1.index[-1].date()}')
print(f'M15 bars: {len(df_m15):,}  |  {df_m15.index[0].date()} → {df_m15.index[-1].date()}')
print()
print('H1 sample:')
df_h1.tail(5)

In [ ]:
# Quick stats — what does the data look like?
print('=== H1 Price Range ===')
print(f'  Min close: {df_h1["Close"].min():.5f}  (historic low)')
print(f'  Max close: {df_h1["Close"].max():.5f}  (historic high)')
print(f'  Avg H1 range (H-L): {(df_h1["High"] - df_h1["Low"]).mean() / 0.0001:.1f} pips')
print()

# Bars per year
df_h1['year'] = df_h1.index.year
bars_per_year = df_h1.groupby('year').size()
print('=== H1 Bars per Year ===')
print(bars_per_year.to_string())

## 2 — Volume Profile Visualisation

The volume profile shows **where the most trading activity happened** at each price level.

- **POC** (Point of Control) — the price level with the highest traded volume. Institutions anchor positions here.
- **HVN** (High Volume Node) — clusters above 70% of peak volume. Price tends to slow and reverse here.
- **LVN** (Low Volume Node) — thin areas below 30% of peak volume. Price moves fast through these — good targets.

In [ ]:
from src.indicators.volume_profile import build_volume_profile

# ── Change these to explore different periods ─────────────────────
WINDOW_DATE  = '2020-07-01'   # any date in the data
WINDOW_BARS  = 500            # how many H1 bars to include in the profile
# ─────────────────────────────────────────────────────────────────

# Find the bar index for the chosen date
target_idx = df_h1.index.searchsorted(pd.Timestamp(WINDOW_DATE, tz='America/Toronto'))
window = df_h1.iloc[max(0, target_idx - WINDOW_BARS):target_idx]

levels = build_volume_profile(window)

print(f'Profile window: {window.index[0].date()} → {window.index[-1].date()}  ({len(window)} bars)')
print(f'POC:  {levels.poc:.5f}')
print(f'HVNs: {len(levels.hvns)} nodes  |  first 5: {[round(h,5) for h in levels.hvns[:5]]}')
print(f'LVNs: {len(levels.lvns)} nodes  |  first 5: {[round(l,5) for l in levels.lvns[:5]]}')

In [ ]:
# ── Plot: price chart + volume profile histogram side by side ─────
# This is what an institutional trader sees when reading the market.

fig = plt.figure(figsize=(18, 8), facecolor=BG)
gs  = gridspec.GridSpec(1, 2, width_ratios=[3, 1], wspace=0.02)
ax_price = fig.add_subplot(gs[0])
ax_vol   = fig.add_subplot(gs[1], sharey=ax_price)

for ax in [ax_price, ax_vol]:
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#2a2a3a')

# Price as a simple line (for clarity)
ax_price.plot(window.index, window['Close'], color=BLUE, linewidth=0.8, alpha=0.9)

# Overlay POC, HVNs, LVNs as horizontal lines
ax_price.axhline(levels.poc, color=ORANGE, linewidth=1.5, linestyle='-',
                 label=f'POC  {levels.poc:.5f}', zorder=5)
for h in levels.hvns:
    ax_price.axhline(h, color=GREEN, linewidth=0.6, linestyle='--', alpha=0.5)
for l in levels.lvns:
    ax_price.axhline(l, color=RED, linewidth=0.4, linestyle=':', alpha=0.4)

# Add dummy lines for legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=ORANGE, linewidth=1.5, label=f'POC  {levels.poc:.5f}'),
    Line2D([0], [0], color=GREEN,  linewidth=1,   linestyle='--', label=f'HVNs ({len(levels.hvns)})'),
    Line2D([0], [0], color=RED,    linewidth=1,   linestyle=':',  label=f'LVNs ({len(levels.lvns)})'),
]
ax_price.legend(handles=legend_elements, fontsize=8,
                facecolor='#1a1a2e', labelcolor=TEXT, framealpha=0.7)
ax_price.set_title(f'EURUSD H1 — Volume Profile  |  {window.index[0].date()} → {window.index[-1].date()}',
                   color=TEXT, fontsize=10)
ax_price.set_ylabel('Price', color=GRAY, fontsize=8)

# Volume histogram (horizontal bar chart)
# Bin the price range into 100 levels, sum volume per bin
price_min = window['Low'].min()
price_max = window['High'].max()
bins = np.linspace(price_min, price_max, Config.PROFILE_BINS + 1)
bin_centres = (bins[:-1] + bins[1:]) / 2
bin_volume  = np.zeros(Config.PROFILE_BINS)

for _, row in window.iterrows():
    lo, hi, vol = row['Low'], row['High'], row['Volume']
    in_bin = (bin_centres >= lo) & (bin_centres <= hi)
    n = in_bin.sum()
    if n > 0:
        bin_volume[in_bin] += vol / n

# Colour: HVN = green, LVN = red, POC = orange
poc_idx = np.argmin(np.abs(bin_centres - levels.poc))
bar_colours = []
for bc in bin_centres:
    if abs(bc - levels.poc) / 0.0001 < Config.POC_ZONE_PIPS:
        bar_colours.append(ORANGE)
    elif any(abs(bc - h) / 0.0001 < Config.POC_ZONE_PIPS for h in levels.hvns):
        bar_colours.append(GREEN)
    elif any(abs(bc - l) / 0.0001 < Config.POC_ZONE_PIPS for l in levels.lvns):
        bar_colours.append(RED)
    else:
        bar_colours.append(GRAY)

ax_vol.barh(bin_centres, bin_volume, height=(price_max - price_min) / Config.PROFILE_BINS * 0.9,
            color=bar_colours, alpha=0.7)
ax_vol.set_title('Volume\nProfile', color=TEXT, fontsize=9)
ax_vol.tick_params(labelleft=False)

plt.suptitle('Volume Profile: price slows and reverses at HVNs (green), moves fast through LVNs (red)',
             color=GRAY, fontsize=9, y=0.02)
plt.tight_layout()
plt.savefig('../data/processed/notebook_volume_profile.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.show()
print('Chart saved.')

## 3 — Regime Detection: What Does NEUTRAL Look Like?

The strategy ONLY fires when the H1 trend is **NEUTRAL**. This means:
- EMA50 and EMA200 are close together (no clear crossover)
- Price is not strongly above or below EMA200
- Price structure is not making clear higher highs or lower lows

In a NEUTRAL regime, POC levels act as magnets. Price bounces between them.
In a BULLISH or BEARISH regime, the trend momentum overpowers the level — price blows through.

In [ ]:
from src.indicators.trend_filter import get_trend_state, calculate_adx

# ── Change these to explore different periods ─────────────────────
REGIME_DATE  = '2020-07-03'   # date of a known good trade
REGIME_BARS  = 300            # H1 bars for trend detection window
# ─────────────────────────────────────────────────────────────────

idx = df_h1.index.searchsorted(pd.Timestamp(REGIME_DATE, tz='America/Toronto'))
regime_window = df_h1.iloc[max(0, idx - REGIME_BARS):idx + 1]

state = get_trend_state(regime_window, 'BUY')
adx   = calculate_adx(regime_window)

print(f'Date: {REGIME_DATE}')
print(f'Trend direction:  {state.direction}  (strength {state.strength}/3)')
print(f'EMA50:            {state.ema50:.5f}')
print(f'EMA200:           {state.ema200:.5f}')
print(f'Price vs EMA200:  {state.price_vs_ema}')
print(f'Price structure:  {state.structure}')
print(f'ADX:              {adx:.1f}  ({"RANGING" if adx < 25 else "TRENDING"} — threshold is 25)')
print()
print(f'Signal allowed?   {"YES" if state.direction == "NEUTRAL" and adx < 25 else "NO"}')

In [ ]:
# ── Plot NEUTRAL vs TRENDING regime across time ────────────────────
# This shows visually why some years have many signals and others have few.

PLOT_YEAR = 2020  # change this to any year
year_mask = df_h1['year'] == PLOT_YEAR
df_year   = df_h1[year_mask].copy()

# Sample every 4th bar for speed (regime changes slowly)
step = 4
directions = []
adx_vals   = []
sample_idx = range(300, len(df_year), step)

for j in sample_idx:
    window_j = df_year.iloc[max(0, j - 300):j + 1]
    if len(window_j) < 50:
        directions.append('UNKNOWN')
        adx_vals.append(0)
        continue
    s = get_trend_state(window_j)
    a = calculate_adx(window_j)
    directions.append(s.direction)
    adx_vals.append(a)

sample_times = df_year.index[list(sample_idx)]

fig, axes = plt.subplots(2, 1, figsize=(16, 8), facecolor=BG, sharex=True)
fig.suptitle(f'Regime Detection — EURUSD H1  {PLOT_YEAR}', color=TEXT, fontsize=11)

for ax in axes:
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#2a2a3a')

# Price
axes[0].plot(df_year.index, df_year['Close'], color=BLUE, linewidth=0.8)
axes[0].set_ylabel('Price', color=GRAY, fontsize=8)

# Shade NEUTRAL periods green, TRENDING periods red
for k, (t, d, a) in enumerate(zip(sample_times, directions, adx_vals)):
    color = GREEN if (d == 'NEUTRAL' and a < 25) else (RED if d != 'UNKNOWN' else GRAY)
    alpha = 0.15 if color == RED else (0.25 if color == GREEN else 0.05)
    next_t = sample_times[k + 1] if k + 1 < len(sample_times) else df_year.index[-1]
    axes[0].axvspan(t, next_t, color=color, alpha=alpha)

# ADX line
axes[1].plot(sample_times, adx_vals, color=ORANGE, linewidth=1.0)
axes[1].axhline(25, color=RED, linewidth=0.8, linestyle='--', alpha=0.7,
                label='ADX = 25 (trending threshold)')
axes[1].fill_between(sample_times, 0, adx_vals,
                     where=[a < 25 for a in adx_vals],
                     color=GREEN, alpha=0.2, label='Ranging (ADX < 25)')
axes[1].fill_between(sample_times, 0, adx_vals,
                     where=[a >= 25 for a in adx_vals],
                     color=RED, alpha=0.2, label='Trending (ADX > 25)')
axes[1].set_ylabel('ADX', color=GRAY, fontsize=8)
axes[1].legend(fontsize=7, facecolor='#1a1a2e', labelcolor=TEXT)

# Legend for price chart
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color=GREEN, alpha=0.4, label='NEUTRAL + ADX < 25 (signals allowed)'),
    Patch(color=RED,   alpha=0.3, label='TRENDING or ADX > 25 (signals blocked)'),
], fontsize=7, facecolor='#1a1a2e', labelcolor=TEXT)

plt.tight_layout()
plt.savefig('../data/processed/notebook_regime.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.show()

## 4 — Trade Signal Walkthrough

Walk through a specific signal step by step. Change the date to any trade from the journal.

In [ ]:
from src.indicators.volume_profile import build_volume_profile, price_near_level
from src.indicators.session_profile import build_multi_session_levels
from src.indicators.trend_filter import calculate_adx, calculate_atr, is_trend_aligned, get_trend_state
from src.strategy.vp_strategy import generate_signal, check_session_confluence, volume_confirms_rejection

# ── Pick a trade date from the journal ────────────────────────────
TRADE_DATE = '2020-07-03 10:00'   # 2020-07-03 10:00 — known winning BUY
# ─────────────────────────────────────────────────────────────────

pip_size = 0.0001
idx = df_h1.index.searchsorted(pd.Timestamp(TRADE_DATE, tz='America/Toronto'))
bar = df_h1.iloc[idx]
signal_df = df_h1.iloc[max(0, idx - 300):idx + 1]
profile_window = df_h1.iloc[max(0, idx - 500):idx]

print(f'=== Signal Walkthrough: {TRADE_DATE} ===')
print(f'Bar: O={bar["Open"]:.5f}  H={bar["High"]:.5f}  L={bar["Low"]:.5f}  C={bar["Close"]:.5f}')
print()

# Step 1: Volume profile
levels = build_volume_profile(profile_window)
price  = float(bar['Close'])
near_poc = price_near_level(price, levels.poc, pip_size)
non_poc_hvns = [h for h in levels.hvns if abs(h - levels.poc) / pip_size > Config.POC_ZONE_PIPS]
near_hvn = any(price_near_level(price, h, pip_size) for h in non_poc_hvns)

print(f'STEP 1 — Level Check')
print(f'  POC: {levels.poc:.5f}  |  Near POC: {near_poc}')
print(f'  HVNs: {len(levels.hvns)}  |  Near extra HVN: {near_hvn}')
print(f'  Result: {"PASS" if near_poc or near_hvn else "FAIL — no signal"}')
print()

# Step 2: ADX
adx = calculate_adx(signal_df)
print(f'STEP 2 — ADX Regime Filter')
print(f'  ADX: {adx:.1f}  |  Threshold: {Config.ADX_THRESHOLD}')
print(f'  Result: {"PASS (ranging)" if adx < Config.ADX_THRESHOLD else "FAIL (trending — skip)"}')
print()

# Step 3: Candle
body = abs(float(bar['Close']) - float(bar['Open']))
upper_wick = float(bar['High']) - max(float(bar['Close']), float(bar['Open']))
lower_wick = min(float(bar['Close']), float(bar['Open'])) - float(bar['Low'])
is_bullish = lower_wick > body * 1.5 and bar['Close'] > bar['Open']
is_bearish = upper_wick > body * 1.5 and bar['Close'] < bar['Open']

print(f'STEP 3 — Rejection Candle')
print(f'  Body:       {body/pip_size:.1f} pips')
print(f'  Lower wick: {lower_wick/pip_size:.1f} pips  (need > body × 1.5 = {body*1.5/pip_size:.1f})')
print(f'  Upper wick: {upper_wick/pip_size:.1f} pips  (need > body × 1.5 = {body*1.5/pip_size:.1f})')
print(f'  Bullish rejection: {is_bullish}  |  Bearish rejection: {is_bearish}')
direction = 'BUY' if is_bullish else ('SELL' if is_bearish else None)
print(f'  Direction: {direction}')
print()

# Step 4: Trend
state = get_trend_state(signal_df, direction or 'BUY')
print(f'STEP 4 — Trend Filter (NEUTRAL only)')
print(f'  Trend: {state.direction}  |  Strength: {state.strength}/3')
print(f'  EMA50 vs EMA200: {"50>200 (bullish)" if state.ema50 > state.ema200 else "50<200 (bearish)"}')
print(f'  Price vs EMA200: {state.price_vs_ema}')
print(f'  Structure: {state.structure}')
print(f'  Result: {"PASS (NEUTRAL)" if state.direction == "NEUTRAL" else "FAIL (trending)"}')
print()

# Step 5: ATR min stop
atr_pips = calculate_atr(signal_df) / pip_size
print(f'STEP 5 — ATR Minimum Stop')
print(f'  H1 ATR(14): {atr_pips:.1f} pips')
print(f'  Min SL:     {atr_pips * Config.MIN_STOP_ATR_MULT:.1f} pips  (40% of ATR)')

print()
print('=== Full signal generation ===')
signal = generate_signal(signal_df, levels, pip_size=pip_size)
if signal:
    print(f'  SIGNAL FIRED: {signal.direction}')
    print(f'  Entry:  {signal.entry:.5f}')
    print(f'  SL:     {signal.stop_loss:.5f}  ({abs(signal.entry - signal.stop_loss)/pip_size:.1f} pips)')
    print(f'  TP:     {signal.take_profit:.5f}  (R:R {signal.rr_ratio}:1)')
    print(f'  Reason: {signal.reason}')
else:
    print('  No signal — check steps above for which filter blocked it')

## 5 — Trade Journal Analysis

Load the last backtest results and analyse performance patterns.

In [ ]:
import os

journal_path = '../data/processed/trade_journal.csv'
if not os.path.exists(journal_path):
    print('No trade journal found. Run scripts/run_backtest.py first.')
else:
    journal = pd.read_csv(journal_path)
    journal['entry_time'] = pd.to_datetime(journal['entry_time'], utc=True)
    journal['year'] = journal['entry_time'].dt.year
    journal['month'] = journal['entry_time'].dt.to_period('M')
    
    print(f'Trades loaded: {len(journal)}')
    print(f'Date range: {journal["entry_time"].min().date()} → {journal["entry_time"].max().date()}')
    print()
    
    wins   = (journal['result'] == 'WIN').sum()
    losses = (journal['result'] == 'LOSS').sum()
    print(f'Win rate:       {wins}/{len(journal)} = {wins/len(journal)*100:.1f}%')
    print(f'Profit factor:  {journal[journal["pips_net"]>0]["pips_net"].sum() / abs(journal[journal["pips_net"]<0]["pips_net"].sum()):.2f}')
    print(f'Avg win:        +{journal[journal["result"]=="WIN"]["pips_net"].mean():.1f} pips')
    print(f'Avg loss:       {journal[journal["result"]=="LOSS"]["pips_net"].mean():.1f} pips')

In [ ]:
if 'journal' in dir() and len(journal) > 0:
    # Year-by-year breakdown
    year_stats = journal.groupby('year').apply(lambda g: pd.Series({
        'trades':   len(g),
        'wins':     (g['result'] == 'WIN').sum(),
        'win_pct':  round((g['result'] == 'WIN').mean() * 100, 1),
        'avg_pips': round(g['pips_net'].mean(), 1),
        'net_pips': round(g['pips_net'].sum(), 1),
    })).reset_index()
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor=BG)
    fig.suptitle('Trade Journal Analysis', color=TEXT, fontsize=12)
    
    for ax in axes.flat:
        ax.set_facecolor(PANEL)
        ax.tick_params(colors=GRAY, labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor('#2a2a3a')
    
    # Trades per year
    axes[0,0].bar(year_stats['year'], year_stats['trades'], color=BLUE, alpha=0.8)
    axes[0,0].set_title('Trades per Year', color=TEXT, fontsize=9)
    axes[0,0].set_ylabel('Count', color=GRAY, fontsize=8)
    
    # Win rate per year
    colors = [GREEN if w >= 40 else RED for w in year_stats['win_pct']]
    axes[0,1].bar(year_stats['year'], year_stats['win_pct'], color=colors, alpha=0.8)
    axes[0,1].axhline(40, color=ORANGE, linewidth=1, linestyle='--', label='40% target')
    axes[0,1].set_title('Win Rate per Year', color=TEXT, fontsize=9)
    axes[0,1].set_ylabel('Win %', color=GRAY, fontsize=8)
    axes[0,1].legend(fontsize=7, facecolor='#1a1a2e', labelcolor=TEXT)
    
    # Avg pips per trade per year
    colors2 = [GREEN if p > 0 else RED for p in year_stats['avg_pips']]
    axes[1,0].bar(year_stats['year'], year_stats['avg_pips'], color=colors2, alpha=0.8)
    axes[1,0].axhline(0, color=GRAY, linewidth=0.8)
    axes[1,0].set_title('Avg Pips per Trade', color=TEXT, fontsize=9)
    axes[1,0].set_ylabel('Pips', color=GRAY, fontsize=8)
    
    # Pips per trade (waterfall)
    pips = journal['pips_net'].values
    bar_colors = [GREEN if p > 0 else RED for p in pips]
    axes[1,1].bar(range(len(pips)), pips, color=bar_colors, alpha=0.8, width=0.8)
    axes[1,1].axhline(0, color=GRAY, linewidth=0.8)
    axes[1,1].set_title('Pips per Trade (chronological)', color=TEXT, fontsize=9)
    axes[1,1].set_xlabel('Trade #', color=GRAY, fontsize=8)
    axes[1,1].set_ylabel('Pips (net)', color=GRAY, fontsize=8)
    
    plt.tight_layout()
    plt.savefig('../data/processed/notebook_journal.png', dpi=130, bbox_inches='tight', facecolor=BG)
    plt.show()
    
    print('\n=== Year by Year ===' )
    print(year_stats.to_string(index=False))

## 6 — Parameter Tuning Sandbox

Change parameters here and re-run the backtest without touching the source files.
This is useful for quick experiments before committing changes.

In [ ]:
# ── Experiment: what happens if we lower the TP target? ────────────
# Monthly income goal = frequent small wins.
# Try MIN_RR = 1.5, MAX_RR = 2.0 instead of 2.0 / 4.0.

# Override config values for this experiment only
Config.MIN_RR_RATIO = 1.5   # was 2.0
Config.MAX_RR_RATIO = 2.0   # was 4.0

print('Config overridden for experiment:')
print(f'  MIN_RR_RATIO: {Config.MIN_RR_RATIO}')
print(f'  MAX_RR_RATIO: {Config.MAX_RR_RATIO}')
print()
print('Now run the full backtest from the terminal:')
print('  python scripts/run_backtest.py')
print()
print('Or run it here (takes ~6 minutes):')
print('  Uncomment the block below and run the cell.')

# Uncomment to run the full backtest inline:
# from src.backtester import run_backtest, print_results, TradingCosts
# costs = TradingCosts(spread_pips=1.2, slippage_pips=0.5)
# result = run_backtest(df=df_h1, df_m15=df_m15, costs=costs,
#                       starting_balance=1000.0, verbose=False)
# print_results(result)

In [ ]:
# ── Restore config to defaults ─────────────────────────────────────
Config.MIN_RR_RATIO = 2.0
Config.MAX_RR_RATIO = 4.0
print('Config restored to defaults.')

## 7 — Next Experiment: M30 Entries

Your goal is regular monthly income — frequent small wins, smooth equity curve.
The current M15 entry approach fires ~2-4 times/year.

**M30 strategy:** resample M15 data to M30 (2 × M15 = 1 M30), use as entry timeframe.
- Less noisy than M15 (bars average out 2 M15 bars)
- More opportunities than H1
- Matches your natural trading timeframe

In [ ]:
# Resample M15 → M30
df_m30 = df_m15.resample('30min').agg({
    'Open':   'first',
    'High':   'max',
    'Low':    'min',
    'Close':  'last',
    'Volume': 'sum',
}).dropna()

print(f'M15 bars: {len(df_m15):,}')
print(f'M30 bars: {len(df_m30):,}  (resampled from M15)')
print()
print('M30 sample:')
df_m30.tail(8)

In [ ]:
# Compare M15 vs M30 candle ranges (M30 should have less noise per bar)
m15_ranges = (df_m15['High'] - df_m15['Low']) / 0.0001
m30_ranges = (df_m30['High'] - df_m30['Low']) / 0.0001

print('Candle range comparison (pips):')
print(f'  M15 avg range: {m15_ranges.mean():.1f} pips')
print(f'  M30 avg range: {m30_ranges.mean():.1f} pips')
print(f'  M30 avg is {m30_ranges.mean()/m15_ranges.mean():.1f}x larger per bar')
print()
print('This means M30 rejections have larger, more meaningful wicks.')
print('Less noise = fewer false signals = better win rate per trade.')